In [1]:
import dotenv
dotenv.load_dotenv()

import os
from logfire.query_client import LogfireQueryClient
from datetime import datetime, timedelta, timezone

read_token = os.getenv('LOGFIRE_TOKEN')
logfire_query_client = LogfireQueryClient(read_token=read_token)

In [2]:
from datetime import datetime, timedelta, timezone

# Question 5. Get Trace IDs with SQ

In [3]:
   trace_rows = logfire_query_client.query_json_rows(
        sql="""
        SELECT
            trace_id,
            start_timestamp,
            duration
        FROM records
        WHERE span_name = 'agent run'
           OR span_name LIKE 'trivia_session%'
        ORDER BY start_timestamp DESC
        LIMIT 2
        """,  min_timestamp=datetime.now(timezone.utc) - timedelta(days=1)
    )

In [4]:
trace_ids = [r['trace_id'] for r in trace_rows['rows']]
trace_ids

['01a197e6db0d3f0f82a4f334f97d26fa', '01a197beb11dde98fb9758d0e7acd502']

In [5]:
trace_id = trace_ids[1]

run_row = logfire_query_client.query_json_rows(
    sql=f"""
    SELECT
        attributes->'pydantic_ai.all_messages' as all_messages,
        attributes->>'gen_ai.aggregated_usage.input_tokens' as input_tokens,
        attributes->>'gen_ai.aggregated_usage.output_tokens' as output_tokens
    FROM records
    WHERE trace_id = '{trace_id}'
      AND (span_name = 'agent run' OR span_name LIKE 'invoke_agent %')
    ORDER BY start_timestamp ASC
    """, min_timestamp=datetime.now(timezone.utc) - timedelta(days=7)
)

In [6]:
all_messages = run_row['rows'][0]
all_messages['all_messages'][0]

{'role': 'user',
 'parts': [{'type': 'text',
   'content': "Let's play 5 easy questions from Science & Nature"}]}

# Question 6. Reconstruct Runs and Calculate Costs

In [7]:
from trace_replay.converter import fetch_trace, TraceData

In [8]:
trace_data = TraceData

In [9]:
trace_data = fetch_trace(trace_id, logfire_query_client)

In [10]:
input_tokens = trace_data.input_tokens
output_tokens = trace_data.output_tokens
print( f"input_token:{input_tokens}, output_token:{output_tokens}")

input_token:1181, output_token:132


In [12]:
model_name = 'gpt-4o-mini'

In [13]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def calculate_cost(model_name, input_tokens, output_tokens):
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost

In [14]:
cost = calculate_cost(model_name, input_tokens, output_tokens)
cost

0.00025635